In [ ]:
# Ensure output directory exists
import os
os.makedirs('notebooks', exist_ok=True)
print('Ensured notebooks/ directory exists')

In [1]:
from supabase import create_client, Client
from dotenv import load_dotenv
import os
from getpass import getpass

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise RuntimeError("SUPABASE_URL and SUPABASE_KEY must be set (via .env or interactive input).")

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print("Client Supabase créé.")

Client Supabase créé.


# Analyse PlayerState — Visualisations

Cette section charge un échantillon des données `PlayerState` depuis Supabase et produit 3 graphiques :

- Distribution de `Damage`
- Évolution de la `MoveSpeed` moyenne en bins de frames
- Scatter `Damage` vs `MoveSpeed` coloré par `Victory`

Assure-toi que le fichier `.env` contient `SUPABASE_URL` et `SUPABASE_KEY` avant d'exécuter.

In [7]:
# Code: charger les données et tracer
from supabase import create_client
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# headless backend for automated image saving
import matplotlib
matplotlib.use('Agg')

load_dotenv()
SUPABASE_URL = os.getenv('SUPABASE_URL')
SUPABASE_KEY = os.getenv('SUPABASE_KEY')

if not SUPABASE_URL or not SUPABASE_KEY:
    raise RuntimeError('Définis SUPABASE_URL et SUPABASE_KEY dans .env avant d\'exécuter cette cellule')

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Récupérer un échantillon raisonnable
ps_resp = supabase.table('PlayerState').select('*').limit(5000).execute()
rows = None
if hasattr(ps_resp, 'data'):
    rows = ps_resp.data
elif isinstance(ps_resp, dict) and 'data' in ps_resp:
    rows = ps_resp['data']
else:
    rows = ps_resp

if not rows:
    raise RuntimeError('Aucune donnée PlayerState retournée')

df = pd.DataFrame(rows)

# Récupérer info de Run pour Victory
runs_resp = supabase.table('Run').select('id, Victory').limit(10000).execute()
runs = runs_resp.data if hasattr(runs_resp, 'data') else (runs_resp['data'] if isinstance(runs_resp, dict) and 'data' in runs_resp else runs_resp)
runs_df = pd.DataFrame(runs) if runs else pd.DataFrame(columns=['id','Victory'])

# merge
if 'id_run' in df.columns and 'id' in runs_df.columns:
    df = df.merge(runs_df, left_on='id_run', right_on='id', how='left')

# Convertir colonnes numériques
numeric_cols = ['Damage','MoveSpeed','Frame','Hearts','Coins','Bombs','Keys','TearRange','ShotSpeed','Luck']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

sns.set(style='whitegrid')

# Plot 1: distribution Damage
plt.figure(figsize=(10,6))
if 'Damage' in df.columns:
    sns.histplot(df['Damage'].dropna(), bins=50, kde=True)
    plt.title('Distribution of Damage')
    plt.savefig('notebooks/figure_damage_hist.png')
    plt.close()

# Plot 2: average MoveSpeed by frame bins
if 'Frame' in df.columns and 'MoveSpeed' in df.columns:
    df['frame_bin'] = (df['Frame'] // 100) * 100
    speed_by_frame = df.groupby('frame_bin')['MoveSpeed'].mean().reset_index()
    plt.figure(figsize=(10,6))
    sns.lineplot(data=speed_by_frame, x='frame_bin', y='MoveSpeed')
    plt.title('Average MoveSpeed by Frame (100-frame bins)')
    plt.xlabel('Frame (bin start)')
    plt.savefig('notebooks/figure_movespeed_line.png')
    plt.close()

# Plot 3: Damage vs MoveSpeed colored by Victory (sample)
if 'MoveSpeed' in df.columns and 'Damage' in df.columns:
    sample = df.sample(n=min(1000, len(df)))
    plt.figure(figsize=(10,6))
    hue = 'Victory' if 'Victory' in df.columns else None
    sns.scatterplot(data=sample, x='MoveSpeed', y='Damage', hue=hue, alpha=0.6)
    plt.title('Damage vs MoveSpeed (sampled)')
    plt.savefig('notebooks/figure_damage_vs_movespeed.png')
    plt.close()

print('Images saved to notebooks/: figure_damage_hist.png, figure_movespeed_line.png, figure_damage_vs_movespeed.png')

FileNotFoundError: [Errno 2] No such file or directory: 'notebooks/figure_damage_hist.png'

In [ ]:
# Reload PlayerState with fallback if Run table is inaccessible, then re-generate figures
from supabase import create_client
import os
from dotenv import load_dotenv
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

load_dotenv()
SUPABASE_URL = os.getenv('SUPABASE_URL')
SUPABASE_KEY = os.getenv('SUPABASE_KEY')
if not SUPABASE_URL or not SUPABASE_KEY:
    raise RuntimeError('Définis SUPABASE_URL et SUPABASE_KEY dans .env avant d\'exécuter')

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
ps_resp = supabase.table('PlayerState').select('*').limit(5000).execute()
rows = ps_resp.data if hasattr(ps_resp, 'data') else (ps_resp['data'] if isinstance(ps_resp, dict) and 'data' in ps_resp else ps_resp)
if not rows:
    print('Aucune donnée PlayerState retournée')
else:
    df = pd.DataFrame(rows)
    # try to get run info, but continue if permission denied
    try:
        runs_resp = supabase.table('Run').select('id, Victory').limit(10000).execute()
        runs = runs_resp.data if hasattr(runs_resp, 'data') else (runs_resp['data'] if isinstance(runs_resp, dict) and 'data' in runs_resp else runs_resp)
        runs_df = pd.DataFrame(runs) if runs else pd.DataFrame(columns=['id','Victory'])
    except Exception as e:
        print('Could not fetch Run table (permission error), continuing without Victory column:', e)
        runs_df = pd.DataFrame(columns=['id','Victory'])

    if 'id_run' in df.columns and 'id' in runs_df.columns and not runs_df.empty:
        df = df.merge(runs_df, left_on='id_run', right_on='id', how='left')

    numeric_cols = ['Damage','MoveSpeed','Frame','Hearts','Coins','Bombs','Keys','TearRange','ShotSpeed','Luck']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    sns.set(style='whitegrid')
    # Plot 1
    if 'Damage' in df.columns:
        plt.figure(figsize=(8,5))
        sns.histplot(df['Damage'].dropna(), bins=50, kde=True)
        plt.title('Distribution of Damage')
        plt.savefig('notebooks/figure_damage_hist.png')
        plt.close()
    # Plot 2
    if 'Frame' in df.columns and 'MoveSpeed' in df.columns:
        df['frame_bin'] = (df['Frame'] // 100) * 100
        speed_by_frame = df.groupby('frame_bin')['MoveSpeed'].mean().reset_index()
        plt.figure(figsize=(8,5))
        sns.lineplot(data=speed_by_frame, x='frame_bin', y='MoveSpeed')
        plt.title('Average MoveSpeed by Frame (100-frame bins)')
        plt.xlabel('Frame (bin start)')
        plt.savefig('notebooks/figure_movespeed_line.png')
        plt.close()
    # Plot 3
    if 'MoveSpeed' in df.columns and 'Damage' in df.columns:
        sample = df.sample(n=min(1000, len(df)))
        plt.figure(figsize=(8,5))
        hue = 'Victory' if 'Victory' in df.columns else None
        sns.scatterplot(data=sample, x='MoveSpeed', y='Damage', hue=hue, alpha=0.6)
        plt.title('Damage vs MoveSpeed (sampled)')
        plt.savefig('notebooks/figure_damage_vs_movespeed.png')
        plt.close()

    print('Figures régénérées et sauvegardées.')

In [6]:
try:
    test_ps = supabase.table('PlayerState').select('*').limit(1).execute()
    print("PlayerState OK")
except Exception as e:
    print("PlayerState KO:", e)

try:
    test_run = supabase.table('Run').select('id, Victory').limit(1).execute()
    print("Run OK")
except Exception as e:
    print("Run KO:", e)


PlayerState OK
Run OK


# Visualisations intégrées

### Distribution de Damage

![Distribution de Damage](figure_damage_hist.png)

### Vitesse moyenne par frames (bins)

![Average MoveSpeed by Frame](figure_movespeed_line.png)

### Damage vs MoveSpeed (échantillon)

![Damage vs MoveSpeed](figure_damage_vs_movespeed.png)
